In [1]:
import pandas as pd

import src
from src.load import DataLoader

In [2]:
pd.set_option("display.max_rows", 1024)
pd.set_option("display.max_colwidth", 256)

In [3]:
dl = DataLoader()

In [4]:
videos = (
    dl.videos(filtered=True)
    .join(dl.channels(), "channel_id")
    .select(
        ["video_id", "channel", "video_likes", "video_views", "video_uploadtime", "video_title"],
    )
    .to_pandas()
)

sents = dl.sentences(filtered=True).join(dl.popbert(filtered=True), "sentence_id").to_pandas()

sents = sents.groupby("video_id", observed=True).agg(
    n_sentences=("video_id", "size"),
    elite=("elite", "mean"),
    pplcentr=("pplcentr", "mean"),
)

videos = videos.merge(sents, on="video_id")

/nix/store/wy58cgzaljgbh6g65l13b4v3sc2zklf6-python3-3.11.10-env/lib/python3.11/site-packages/ibis/expr/types/relations.py:685: FutureWarning: Selecting/filtering arbitrary expressions in `Table.__getitem__` is deprecated and will be removed in version 10.0. Please use `Table.select` or `Table.filter` instead.
  warnings.warn(


# Most Liked

In [5]:
top_like_videos = videos[videos.channel != "FDP"]

quantiles = top_like_videos.groupby("channel").video_likes.quantile(q=0.99).rename("quantile")

In [6]:
df = top_like_videos.merge(quantiles, how="left", on="channel")
df["top_1p"] = df.apply(lambda x: 1 if x.video_likes > x["quantile"] else 0, axis=1)

In [7]:
top_videos = (
    df.sort_values(["channel", "video_likes"], ascending=False)
    .groupby("channel")
    .head(
        10,
    )
    .set_index(["channel", "video_id"])
)

In [8]:
top_videos.reset_index()[["channel", "video_likes", "video_views", "video_title"]].to_csv(
    src.OUT / "tables/most_liked_videos_per_channel.csv",
    index=False,
)

# Most Viewed

In [9]:
quantiles = top_videos.groupby("channel").video_views.quantile(q=0.99).rename("quantile")

In [10]:
df = top_like_videos.merge(quantiles, how="left", on="channel")
df["top_1p"] = df.apply(lambda x: 1 if x.video_views > x["quantile"] else 0, axis=1)

In [11]:
top_videos = (
    df.sort_values(["channel", "video_views"], ascending=False)
    .groupby("channel")
    .head(
        10,
    )
    .set_index(["channel", "video_id"])
)

In [12]:
top_videos.reset_index()[["channel", "video_likes", "video_views", "video_title"]].to_csv(
    src.OUT / "tables/most_viewed_videos_per_channel.csv",
    index=False,
)

# Most Anti-Elitism

In [13]:
quantiles = videos.groupby("channel").elite.quantile(q=0.99).rename("quantile")

In [14]:
df = videos.merge(quantiles, how="left", on="channel")
df["top_1p"] = df.apply(lambda x: 1 if x.elite > x["quantile"] else 0, axis=1)

In [15]:
top_videos = (
    df.sort_values(["channel", "elite"], ascending=False)
    .groupby("channel")
    .head(
        10,
    )
    .set_index(["channel", "video_id"])
)

In [16]:
top_videos.reset_index()[["channel", "n_sentences", "elite", "video_title"]].to_csv(
    src.OUT / "tables/most_antielitism_videos_per_channel.csv",
    index=False,
)